In [49]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_openai.llms import OpenAI
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI 
from langchain.messages import HumanMessage


In [17]:
load_dotenv(override=True)

True

In [18]:
loader = PyPDFLoader("Ramatoulaye_Diawane_CV-Pro.pdf")

In [19]:
tokennizer = tiktoken.encoding_for_model("gpt-4o-mini")

In [20]:
print(tokennizer.name)

o200k_base


In [28]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name=tokennizer.name,
    chunk_size=300,
    chunk_overlap=2,
    )

In [29]:
chunks = loader.load_and_split(splitter)

In [30]:
print(len(chunks))

3


In [24]:
print(chunks[0].metadata)

{'producer': '', 'creator': 'WPS Writer', 'creationdate': '2026-03-28T11:10:36+01:00', 'author': '', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-03-28T11:10:36+01:00', 'sourcemodified': "D:20260328111036+01'00'", 'subject': '', 'title': 'Ramatoulaye DIAWANE', 'trapped': '/False', 'source': 'Ramatoulaye_Diawane_CV-Pro.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


In [31]:
embeddings_model = OpenAIEmbeddings()

In [32]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    collection_name="CV_data_collection"
)

In [33]:
retriever = vector_store.as_retriever(kwargs= {"k":10})

In [51]:
## Searching information about cantidates in the resume
@tool
def retriever_tool(query : str) -> str:
     """
     Permet de chercher des informations sur des candidats:
     -Nom, Prénom, Diplômes
     -Expériences professionnelles
     -Compétences techniques
     """
     relevent_chunks=retriever.invoke(query)
     context_list = [d.page_content for d in relevent_chunks]
     context = ".".join(context_list)
     return context



In [59]:
@tool
def get_company_info(company_name : str):
    """
    Consulter des informations sur l'entreprise donnée
    """
    return {
        "company_name": company_name,
        "domaine": "IT",
        "turnover":6000

    }
    

In [60]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_agent(
    model=llm,
    tools=[retriever_tool, get_company_info],
    system_prompt="Réponds à la question de l'utilisateur en utilisant les tools fournies. "

)

In [63]:
resp = agent.invoke(input={
    "messages":[
        HumanMessage("Nom, prénom, diplômes de Ramatoulaye Diawane et les information sur son entreprise") 
    ]
})

In [64]:
print(resp['messages'][-1].content)

### Informations sur Ramatoulaye Diawane

- **Nom**: Diawane
- **Prénom**: Ramatoulaye
- **Diplômes**:
  - Cycle d'Ingénieur d'État en Informatique & IA (en cours, HESTIM – Engineering & Business School, Casablanca, Maroc)
  - Licence Professionnelle en Génie des Données et Technologies Omics (2023 - 2024, École Supérieure Polytechnique, Dakar, Sénégal)
  - Baccalauréat S, Série Scientifique S2 (2018 - 2019, Lycée Seydina Limamou Laye, Dakar, Sénégal)

### Compétences Techniques
- **Programmation**: Python, JavaScript, Bash, SQL
- **Data & Analyse**: ETL, Analyse et visualisation de données
- **Machine Learning / AI**: Supervisé et non-supervisé, Agentic AI
- **Outils**: Git & GitHub, Docker
- **Cloud & Big Data**: AWS, Azure, Notions de Spark, Kafka
- **Développement & APIs**: Django, React, Rest APIs
- **Systèmes, Réseaux & Cybersecurité**: Linux, VM

### Expériences Professionnelles
- **Stagiaire bio-informatique** à l'Institut de Recherche En Santé de Surveillance Épidémiologique e

In [65]:
from IPython.display import Markdown

In [66]:
print(display(Markdown(resp['messages'][-1].content)))

### Informations sur Ramatoulaye Diawane

- **Nom**: Diawane
- **Prénom**: Ramatoulaye
- **Diplômes**:
  - Cycle d'Ingénieur d'État en Informatique & IA (en cours, HESTIM – Engineering & Business School, Casablanca, Maroc)
  - Licence Professionnelle en Génie des Données et Technologies Omics (2023 - 2024, École Supérieure Polytechnique, Dakar, Sénégal)
  - Baccalauréat S, Série Scientifique S2 (2018 - 2019, Lycée Seydina Limamou Laye, Dakar, Sénégal)

### Compétences Techniques
- **Programmation**: Python, JavaScript, Bash, SQL
- **Data & Analyse**: ETL, Analyse et visualisation de données
- **Machine Learning / AI**: Supervisé et non-supervisé, Agentic AI
- **Outils**: Git & GitHub, Docker
- **Cloud & Big Data**: AWS, Azure, Notions de Spark, Kafka
- **Développement & APIs**: Django, React, Rest APIs
- **Systèmes, Réseaux & Cybersecurité**: Linux, VM

### Expériences Professionnelles
- **Stagiaire bio-informatique** à l'Institut de Recherche En Santé de Surveillance Épidémiologique et de Formation (IRESSEF), Diamnadio, Sénégal (2023-01 à 2025-07)
  - Analyse de données génomiques
  - Aide avec les pipelines d'analyse de données bio-informatiques
  - Création de Dashboard pour intégrer les données analysées

### Informations sur l'entreprise
- **Nom de l'entreprise**: Ramatoulaye Diawane
- **Domaine**: IT
- **Chiffre d'affaires**: 6000

### Contact
- **E-mail**: r.diawane@hestim.ma
- **LinkedIn**: Ramatoulaye Diawane
- **Téléphone**: 212 065 643
- **Adresse**: Casablanca Settat

### Langues
- Français (courant)
- Anglais (intermédiaire)

### Centres d'intérêt
- Lectures
- Voyages
- Documentaires
- Recherches

Ramatoulaye Diawane est une étudiante passionnée par la technologie et l'innovation, cherchant à participer à des projets concrets pour mettre en pratique ses acquis.

None
